# Export QC'd daily rainfall to Station Exchange Format (SEF)

The located, quality-controlled daily rainfall observations are shared with
others in the [Station Exchange Format](https://datarescue.climate.copernicus.eu/station-exchange-format-sef)
(SEF): a simple tab-separated text format where **one file holds one variable
from one station**.

Here each ensemble transcription file is a single *station-year* of daily
rainfall. The ensemble frequently holds **duplicate** transcriptions of the same
station-year, so the export **merges** duplicates (exact matches sharing a
location and year) into **one SEF `.tsv` per real station-year** under
`<output_root>/tsv/<year>/<ID>.tsv`. For every day the value is taken from the
duplicate with the best QC verdict (`qc1=pass` &gt; `qc1=fail & qc2=pass` &gt;
`qc1=review` &gt; `qc1=fail & qc2=indeterminate` &gt; the rest), so no day is
dropped. Each file carries every day's consensus daily total (the member
median), converted from the original inches to millimetres, with the QC verdicts
and the contributing `source=` travelling in each observation's `Meta` column
(`qc1=...` from the exact-monthly check, `qc2=...` from the secondary XGBoost
check); the file-level `Meta` lists every merged source.

This notebook runs a small local export, inspects a generated file, validates
its structure, and shows how the same export runs at scale on SLURM.


## Setup roots and imports

In [1]:
import os
from pathlib import Path

from src.rainfall_rescue_sqlite.sef_export import (
    export_sef,
    default_sef_output_root,
    SEF_VERSION,
    VBL,
    STAT,
    UNITS,
    PERIOD,
    INCHES_TO_MM,
    HEADER_ORDER,
    DATA_COLUMNS,
)
from src.rainfall_rescue_sqlite.parquet_regional_stats import (
    default_daily_consensus_parquet_root,
)
from src.rainfall_rescue_sqlite.parquet_similarity import (
    default_comparison_parquet_root,
)

# Cap DuckDB memory and give it a disk spill dir so the local run cannot OOM the
# workstation (the ordered join spills to disk instead of crashing).
os.environ.setdefault("PDIR", "/data/scratch/philip.brohan/ADRQ")
os.environ.setdefault("DUCKDB_MEMORY_LIMIT", "3GB")
os.environ.setdefault("DUCKDB_TEMP_DIR", "/var/tmp/duckdb_sef_export")
Path(os.environ["DUCKDB_TEMP_DIR"]).mkdir(parents=True, exist_ok=True)

pdir = Path(os.environ["PDIR"])
comparison_root = default_comparison_parquet_root()
consensus_root = default_daily_consensus_parquet_root()

print(f"PDIR:            {pdir}")
print(f"comparison root: {comparison_root}")
print(f"consensus root:  {consensus_root}")
print(f"SEF output root: {default_sef_output_root()}")
print(f"SEF version:     {SEF_VERSION}  (Vbl={VBL}, Stat={STAT}, Units={UNITS}, Period={PERIOD})")

PDIR:            /data/scratch/philip.brohan/ADRQ
comparison root: /data/scratch/philip.brohan/ADRQ/monthly_similarity_parquet
consensus root:  /data/scratch/philip.brohan/ADRQ/daily_consensus_parquet
SEF output root: /data/scratch/philip.brohan/ADRQ/sef_export
SEF version:     1.0.0  (Vbl=rr, Stat=sum, Units=mm, Period=1day)


## Run a small local export

Export a couple of matched years into a demo directory under `/var/tmp`.
`export_sef` streams the consensus/metadata/QC join ordered by
`(matched_year, group_key)` and flushes one `.tsv` each time that key advances.
Because the ensemble often contains **duplicate** transcriptions of the same
station-year, all of a station's duplicates (exact matches sharing a location and
year) arrive together and are **merged**: for every day the value is taken from
the duplicate with the best QC verdict, and the source it came from is recorded.
Memory stays bounded to one station-year at a time.


In [2]:
demo_root = Path("/var/tmp/sef_demo")

# Export a couple of years into a demo directory. Sharding is by matched year so
# that every duplicate transcription of a station-year is merged into one file.
result = export_sef(
    output_root=demo_root,
    start_year=1869,
    end_year=1870,
)
result


SEFExportResult(output_root=PosixPath('/var/tmp/sef_demo'), qc_session_id=3, files_written=50, obs_rows=18600, start_year=1869, end_year=1870)

## Inspect a generated SEF file

Each file has 12 header lines (`name<TAB>value`), then the data-table column
header, then one line per observed day.

In [3]:
tsv_files = sorted(demo_root.glob("tsv/*/*.tsv"))
print(f"{len(tsv_files)} SEF files written to {demo_root / 'tsv'}")

sample = tsv_files[0]
lines = sample.read_text(encoding="utf-8").splitlines()
print(f"\nSample file: {sample.relative_to(demo_root)}  ({len(lines)} lines)\n")
print("\n".join(lines[:21]))
print("...")

50 SEF files written to /var/tmp/sef_demo/tsv

Sample file: tsv/1869/DRain_1861-1870_Anglesey-3.tsv  (385 lines)

SEF	1.0.0
ID	DRain_1861-1870_Anglesey-3
Name	LLANGADWALADR BODORGAN
Lat	53.1792
Lon	-4.4166
Alt	30.5
Source	RainfallRescue
Link	NA
Vbl	rr
Stat	sum
Units	mm
Meta	orig.units=in|match.type=exact|qc.session=3|n.sources=1|sources=DRain_1861-1870_Anglesey-3
Year	Month	Day	Hour	Minute	Period	Value	Meta
1869	1	1	9	0	1day	0.0	qc1=fail|qc2=indeterminate|source=DRain_1861-1870_Anglesey-3
1869	1	2	9	0	1day	24.1	qc1=fail|qc2=indeterminate|source=DRain_1861-1870_Anglesey-3
1869	1	3	9	0	1day	17.8	qc1=fail|qc2=indeterminate|source=DRain_1861-1870_Anglesey-3
1869	1	4	9	0	1day	14.2	qc1=fail|qc2=indeterminate|source=DRain_1861-1870_Anglesey-3
1869	1	5	9	0	1day	17.8	qc1=fail|qc2=indeterminate|source=DRain_1861-1870_Anglesey-3
1869	1	6	9	0	1day	6.9	qc1=fail|qc2=indeterminate|source=DRain_1861-1870_Anglesey-3
1869	1	7	9	0	1day	8.6	qc1=fail|qc2=indeterminate|source=DRain_1861-1870_Anglesey-3
1869

## Validate the file structure

Confirm the 12 header lines are in the required order, the data-table header is
correct, every day round-trips back as a row, and the QC verdicts are present.

In [4]:
import pandas as pd

header = dict(line.split("\t", 1) for line in lines[:12])
assert list(header) == HEADER_ORDER, list(header)
assert lines[12].split("\t") == DATA_COLUMNS, lines[12]
# The file-level Meta records the merged sources (n.sources + sources list).
assert "n.sources=" in header["Meta"] and "sources=" in header["Meta"], header["Meta"]

df = pd.read_csv(sample, sep="\t", skiprows=12)
assert len(df) == len(lines) - 13, (len(df), len(lines))
assert (df["Value"] >= 0).all()
# Each observation records its QC verdicts and the source it was taken from.
assert df["Meta"].str.startswith("qc1=").all()
assert df["Meta"].str.contains("|source=", regex=False).all()

print(f"Station:    {header['ID']}  ({header['Name']})")
print(f"Location:   lat={header['Lat']}, lon={header['Lon']}, alt={header['Alt']} m")
print(f"File Meta:  {header['Meta']}")
print(f"Days:       {len(df)}")
print(df["Value"].describe().round(2).to_string())


Station:    DRain_1861-1870_Anglesey-3  (LLANGADWALADR BODORGAN)
Location:   lat=53.1792, lon=-4.4166, alt=30.5 m
File Meta:  orig.units=in|match.type=exact|qc.session=3|n.sources=1|sources=DRain_1861-1870_Anglesey-3
Days:       372
count    372.00
mean       4.32
std        5.80
min        0.00
25%        0.00
50%        2.50
75%        6.68
max       24.10


### QC verdict distribution across the demo export

`qc1` is the exact-monthly check verdict for every day; `qc2` is the secondary
XGBoost verdict, which only exists (`pass`/`fail`/`indeterminate`) for the days
that failed `qc1` and were re-examined — it is `NA` otherwise.

In [5]:
from collections import Counter


def _qc_combo(meta: str) -> str:
    """Return just the ``qc1=...|qc2=...`` part of an observation's Meta."""
    parts = dict(kv.split("=", 1) for kv in meta.split("|") if "=" in kv)
    return f"qc1={parts.get('qc1', 'NA')}|qc2={parts.get('qc2', 'NA')}"


combos = Counter()
for path in tsv_files:
    for line in path.read_text(encoding="utf-8").splitlines()[13:]:
        combos[_qc_combo(line.rsplit("\t", 1)[-1])] += 1

total = sum(combos.values())
print(f"{total} observations across {len(tsv_files)} files\n")
for meta, count in sorted(combos.items()):
    print(f"{count:9d}  ({100 * count / total:5.1f}%)  {meta}")


18600 observations across 50 files

      183  (  1.0%)  qc1=fail|qc2=fail
     9021  ( 48.5%)  qc1=fail|qc2=indeterminate
     3940  ( 21.2%)  qc1=fail|qc2=pass
     5456  ( 29.3%)  qc1=pass|qc2=NA


## Running at scale on SLURM

The full station-year export runs as a single SLURM array (no merge stage —
each shard writes disjoint year-partitioned `.tsv` files):

```bash
scripts/slurm/submit_sef_export.sh
```

Each array task runs `scripts/export_sef.py` for a contiguous **matched-year**
slice (`--num-shards` / `--shard-index` / `--min-year` / `--max-year`). Sharding
by year keeps every duplicate of a station-year in the same task so they can be
merged. Shard count and resources are configured in `scripts/slurm/config.sh`
(the `SEF_*` block: `SEF_NUM_SHARDS`, `SEF_MIN_YEAR`, `SEF_MAX_YEAR`,
`SEF_SOURCE`, `SEF_LINK`, `SEF_OBS_HOUR`, `SEF_CORES` / `SEF_MEM_MB` /
`SEF_TIME_MIN`). Set `SEF_MIN_YEAR` / `SEF_MAX_YEAR` to the min/max
`matched_year` in `ensemble_metadata`.

Prerequisites are the daily-consensus table (`submit_daily_consensus.sh`) and
the located ensemble metadata (`assign_ensemble_metadata`); the QC tables are
joined when present. Files land in
`$PDIR/sef_export/tsv/<year>/<ID>.tsv` with per-shard manifests in
`$PDIR/sef_export/manifests`.

The demo output above lives under `/var/tmp/sef_demo`; remove it with
`rm -rf /var/tmp/sef_demo` when finished.
